# Production keeper contracts


These smoke tests cover the production core at the feature-boundary level. They are intentionally small so failures point to one keeper behavior at a time.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO
import asyncio
import json
from pathlib import Path

from fastcore.nbio import mk_cell, new_nb, read_nb, write_nb as write_raw_nb

from nbskill.execute import exec_nb
from nbskill.foundation import demo_path, remove_demo_path, stamp_notebook_metadata
from nbskill.mcp import create_mcp
from nbskill.parallel import execution_slot, notebook_key
from nbskill.read import nb_cell, nb_chapter, nb_overview, show_doc
from nbskill.review import diff_nb, notebook_validation_problems, style_report
from nbskill.write import batch_edit_nb, update_cell, write_nb


## Shared fixture


In [ ]:
root = demo_path("test_production_contracts")
root.mkdir(parents=True, exist_ok=True)
nb_path = root / "keeper.ipynb"
nb = new_nb([
    mk_cell("# Keeper notebook", cell_type="markdown"),
    mk_cell("## Math", cell_type="markdown"),
    mk_cell("#| export\ndef add(a, b):\n    '''Add two numbers.'''\n    return a + b", cell_type="code"),
    mk_cell("assert add(1, 2) == 3", cell_type="code"),
])
stamp_notebook_metadata(nb)
write_raw_nb(nb, nb_path)
assert nb_path.exists()


## Reading contracts


In [ ]:
out = StringIO()
with redirect_stdout(out): nb_overview(str(nb_path))
assert "def add(a, b):" in out.getvalue()


In [ ]:
out = StringIO()
with redirect_stdout(out): nb_chapter(str(nb_path), name="Math")
assert "## Math" in out.getvalue()


In [ ]:
out = StringIO()
with redirect_stdout(out): nb_cell(str(nb_path), query='contains="def add"')
assert "1 | #| export" in out.getvalue()


In [ ]:
out = StringIO()
with redirect_stdout(out): show_doc(str(nb_path), "add")
assert "Add two numbers." in out.getvalue()


## Writing contracts


In [ ]:
write_nb(str(nb_path), "%%markdown\n## Result\n", after_id=read_nb(nb_path).cells[-1].id)
assert any(cell.source.startswith("## Result") for cell in read_nb(nb_path).cells)


In [ ]:
target = next(cell for cell in read_nb(nb_path).cells if "return a + b" in cell.source)
update_cell(str(nb_path), "return a + b + 0", cell_id=target.id, old_str="return a + b")
assert "return a + b + 0" in next(cell.source for cell in read_nb(nb_path).cells if cell.id == target.id)


In [ ]:
target = next(cell for cell in read_nb(nb_path).cells if cell.source.startswith("assert add"))
plan = {"operations": [{"op": "set_cell_source", "path": str(nb_path), "cell_id": target.id, "source": "assert add(2, 3) == 5"}]}
batch_edit_nb(json.dumps(plan), dry_run=False)
assert "assert add(2, 3) == 5" in next(cell.source for cell in read_nb(nb_path).cells if cell.id == target.id)


## Execution and review contracts


In [ ]:
out = StringIO()
with redirect_stdout(out): exec_nb(str(nb_path), safe=False, show_output=True, timeout=5)
assert "5" in out.getvalue() or out.getvalue() == ""


In [ ]:
assert notebook_validation_problems(str(nb_path)) == []


In [ ]:
report = style_report(str(nb_path), max_output_chars=400, max_diagnostics=5)
assert "summary" in report


In [ ]:
out = StringIO()
with redirect_stdout(out): diff_nb(str(nb_path), ref_a=None)
assert "No code cell changes" in out.getvalue() or "assert add" in out.getvalue()


## MCP and concurrency contracts


In [ ]:
assert notebook_key(nb_path).endswith("keeper.ipynb")


In [ ]:
with execution_slot():
    inside_execution_slot = True
assert inside_execution_slot


In [ ]:
mcp = create_mcp()
tools = {tool.name for tool in asyncio.run(mcp.list_tools())}
assert {"nb_overview", "nb_cell", "exec_nb", "diff_nb", "doctor", "style_check"} <= tools


In [ ]:
remove_demo_path(root)
assert not root.exists()
